# Database
### Criminal Justice Schema — Real Data from CSV + Excel

**Files used:**
- `105_1_regression_data_en.csv` — 20,531 homicide cases (2013–2019)
- `Registru.xlsx` — Russian regional statistics (2012–2019), 76 regions

**Target schema:** `project`

---

### Data $\rightarrow$ Table mapping

| Source | Tables populated |
|---|---|
| CSV rows | `person` (offenders, victims, judges), `person_status_history`, `court`, `case_file`, `sentence`, `crime_act`, `crime_motive`, `case_participant`, `case_judge`, `offender_risk_assessment` |
| Excel rows | `region_statistics` |
| Both (names) | `spatial_hierarchy` (regions from Excel, cities/courts from CSV) |
| Synthetic | `police_department`, `police_officer`, `prison`, `investigation`, `investigation_officer`, `evidence`, `appeal`, `appeal_judge`, `case_location`, `sentence_prison`, `police_deployment`, `role`, `case_status` |

### CSV column mapping
| CSV column | Maps to |
|---|---|
| `ID` | `case_file.case_id` (hashed) |
| `gender_victim` | `person.gender` (victim) |
| `gender_judge` | `person.gender` (judge) |
| `gender_offender` | `person.gender` (offender) |
| `recidivism_type` | `offender_risk_assessment` + `case_participant` notes |
| `guilty_plea` | `case_file.guilty_plea` |
| `soft_binary` | `crime_motive.description` (mitigating) |
| `hard_binary` | `crime_act.description` (aggravating) |
| `rel_type` | `crime_act.act_type` |
| `year` | `case_file.year` + `report_date` |
| `children` | `person.has_children` (offender) |
| `sentence_period_whole` | `sentence.total_duration_months` |
| `region` | `spatial_hierarchy` (region lookup) |
| `court_name` | `court.name` + `spatial_hierarchy` (city) |

In [ ]:
# install dependencies
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'pandas', 'openpyxl', 'faker', 'sqlalchemy', 'psycopg2-binary', '--quiet'
], check=True)

## Import libraries

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import date, timedelta
import random
import time
from sqlalchemy import create_engine, text, event
from sqlalchemy.exc import OperationalError

In [ ]:
fake    = Faker('en_US')
fake_ru = Faker('ru_RU')
random.seed(42)
np.random.seed(42)

# file paths — adjust if needed
CSV_PATH   = '105_1_regression_data_en.csv'
EXCEL_PATH = 'Registru.xlsx'

## Database Connection

In [ ]:
DB_HOST     = 'server_x'
DB_PORT     = 5432
DB_NAME     = 'your_database_name_here'
DB_USER     = 'your_username_here'
DB_PASSWORD = 'insert_your_password_here'
SCHEMA      = 'project'

from sqlalchemy import create_engine, text
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    connect_args={'options': f'-csearch_path={SCHEMA}'}
)
try:
    with engine.connect() as conn:
        conn.execute(text('SELECT 1'))
    print('Connected to database')
except Exception as e:
    print(f'Connection failed: {e}')
    print('Running in PREVIEW MODE (DataFrames only, no DB writes)')

Connected to database


## Load Source Files

In [ ]:
# load CSV
df_csv = pd.read_csv(CSV_PATH, index_col=0)
print(f'CSV loaded: {len(df_csv):,} rows × {len(df_csv.columns)} columns')

# load excel
raw_xl = pd.read_excel(EXCEL_PATH, sheet_name='Foaie1', header=0)
YEARS     = [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
# Row 0 = year header labels; rows 1+ = region data
regions_raw = raw_xl.iloc[1:, 0].reset_index(drop=True)   # Cyrillic region names
print(f'Excel loaded: {len(regions_raw)} regions × {len(YEARS)} years')

#  extract 8-column metric block from Excel
def extract_metric(df, col_start, years):
    cols  = list(range(col_start, col_start + len(years)))
    block = df.iloc[1:, cols].reset_index(drop=True)
    block.columns = years
    return block

xl_article105   = extract_metric(raw_xl, 1,  YEARS)
xl_mean_pop     = extract_metric(raw_xl, 9,  YEARS)
xl_murders_1000 = extract_metric(raw_xl, 17, YEARS)
xl_gini         = extract_metric(raw_xl, 25, YEARS)
xl_unempl       = extract_metric(raw_xl, 33, YEARS)
xl_clearance    = extract_metric(raw_xl, 41, [2013,2014,2015,2016,2017,2018,2019])

def safe_num(val, decimals=3):
    """return None if NaN/unconvertible, else round to decimals"""
    if pd.isna(val): return None
    try:   return round(float(val), decimals)
    except: return None

CSV loaded: 20,531 rows × 14 columns
Excel loaded: 76 regions × 8 years


## spatial_hierarchy

- **Russia** → root country (location_id = 1)
- **76 regions** from Excel → children of Russia (+1 region: `Другие регионы` from CSV)
- **Courts as cities** → unique `court_name` values from CSV become `type='city'` entries, linked to their region

In [ ]:
spatial_rows = []

# russia root
spatial_rows.append({
    'location_id': 1,
    'name': 'Russia',
    'type': 'country',
    'postal_code': None,
    'parent_location_id': None
})

# regions from Excel (+ catch-all from CSV)
# build region name -> location_id map for later lookups
region_name_to_id = {}
loc_id = 2

for i, name in enumerate(regions_raw):
    region_name_to_id[str(name)] = loc_id
    spatial_rows.append({
        'location_id': loc_id,
        'name': str(name),
        'type': 'region',
        'postal_code': str(100000 + i * 1000),
        'parent_location_id': 1
    })
    loc_id += 1

# catch-all region present in CSV but not Excel
CATCHALL_REGION = 'Другие регионы'
region_name_to_id[CATCHALL_REGION] = loc_id
spatial_rows.append({
    'location_id': loc_id,
    'name': CATCHALL_REGION,
    'type': 'region',
    'postal_code': None,
    'parent_location_id': 1
})
loc_id += 1

# courts as cities: unique court_name from CSV
# each court gets a city-type spatial entry under its region
court_city_map = {}  # court_name → location_id
csv_courts = df_csv[['court_name', 'region']].drop_duplicates().reset_index(drop=True)

for _, row in csv_courts.iterrows():
    cname  = row['court_name']
    rname  = row['region']
    parent = region_name_to_id.get(rname, region_name_to_id[CATCHALL_REGION])
    court_city_map[cname] = loc_id

    
    spatial_rows.append({
        'location_id': loc_id,
        'name': cname,
        'type': 'city',         # court location -> city-level entry
        'postal_code': None,
        'parent_location_id': parent
    })
    loc_id += 1

df_spatial = pd.DataFrame(spatial_rows)
print(f'spatial_hierarchy: {len(df_spatial):,} rows')
print(f'1 country + {len(regions_raw)+1} regions + {len(csv_courts)} court-cities')
df_spatial.head(3)

spatial_hierarchy: 2,098 rows
1 country + 77 regions + 2020 court-cities


,location_id,name,type,postal_code,parent_location_id
0,1,Russia,country,NaN,NaN
1,2,Алтайский край,region,100000,1.0
2,3,Амурская область,region,101000,1.0


## region_statistics (from Excel)

Real data for all 76 regions × 8 years. Temperatures are synthetic (realistic for Russia).

In [32]:
stat_rows = []
for r_idx, region_name in enumerate(regions_raw):
    region_id = region_name_to_id[str(region_name)]
    for y_idx, year in enumerate(YEARS):
        art105   = safe_num(xl_article105.iloc[r_idx, y_idx], 0)
        pop      = safe_num(xl_mean_pop.iloc[r_idx, y_idx], 0)
        murd     = safe_num(xl_murders_1000.iloc[r_idx, y_idx])
        gini_val = safe_num(xl_gini.iloc[r_idx, y_idx])
        unemp    = safe_num(xl_unempl.iloc[r_idx, y_idx])
        if year == 2012:
            clear = None
        else:
            ci    = [2013,2014,2015,2016,2017,2018,2019].index(year)
            clear = safe_num(xl_clearance.iloc[r_idx, ci])

        stat_rows.append({
            'region_id':                  region_id,
            'year':                       year,
            'mean_population':            int(pop) if pop is not None else None,
            'total_article_105_cases':    int(art105) if art105 is not None else None,
            'murders_per_1000':           murd,
            'gini_index':                 gini_val,
            'unemployment_rate':          unemp,
            'clearance_rate':             clear,
            'mean_january_temperature':   round(random.uniform(-25, 2), 2),
            'mean_june_temperature':      round(random.uniform(12, 28), 2),
        })

df_region_stats = pd.DataFrame(stat_rows)
print(f'region_statistics: {len(df_region_stats):,} rows  ({len(regions_raw)} regions × {len(YEARS)} years)')
df_region_stats.head(3)

region_statistics: 608 rows  (76 regions × 8 years)


,region_id,year,mean_population,total_article_105_cases,murders_per_1000,gini_index,unemployment_rate,clearance_rate,mean_january_temperature,mean_june_temperature
0,2,2012,NaN,NaN,0.100,0.377,6.2,NaN,-7.74,12.40
1,2,2013,2394694.0,205.0,0.086,0.379,8.3,7.729,-17.57,15.57
2,2,2014,2387725.0,199.0,0.083,0.381,7.2,8.213,-5.12,22.83


## case_status, role

In [33]:
# case_status
case_status_names = ['Open', 'Closed', 'Pending', 'Appealed', 'Dismissed']
df_case_status    = pd.DataFrame([
    {'case_status_id': i+1, 'name': s} for i, s in enumerate(case_status_names)
])
status_to_id = {row['name']: row['case_status_id'] for _, row in df_case_status.iterrows()}
print(f'case_status: {len(df_case_status)} rows => {status_to_id}')

# role
roles_data = [
    (1, 'Defendant',        'The accused party in criminal proceedings', 'Article 47 CPC'),
    (2, 'Victim',           'The harmed party',                          'Article 42 CPC'),
    (3, 'Witness',          'Provides testimony',                        'Article 56 CPC'),
    (4, 'Prosecutor',       'State representative',                      'Article 37 CPC'),
    (5, 'Defense Attorney', 'Legal representative of defendant',         'Article 49 CPC'),
    (6, 'Expert Witness',   'Provides specialized knowledge',            'Article 57 CPC'),
    (7, 'Civil Plaintiff',  'Claims civil damages',                      'Article 44 CPC'),
    (8, 'Interpreter',      'Language assistance',                       'Article 59 CPC'),
    (9, 'Legal Guardian',   'Represents minor/incapacitated',            'Article 48 CPC'),
    (10,'Accomplice',       'Participated in crime',                     'Article 33 CC'),
]
df_role = pd.DataFrame(roles_data, columns=['role_id','role_name','role_description','legal_authority'])
ROLE_DEFENDANT = 1
ROLE_VICTIM    = 2
print(f'role: {len(df_role)} rows')

case_status: 5 rows => {'Open': 1, 'Closed': 2, 'Pending': 3, 'Appealed': 4, 'Dismissed': 5}
role: 10 rows


## court (from CSV court_name)

1,924 unique courts extracted directly from the CSV.

In [34]:
court_types = ['Criminal', 'Civil', 'Administrative', 'Military', 'District', 'Regional']

# unique courts with their region context
unique_courts = df_csv[['court_name', 'region']].drop_duplicates('court_name').reset_index(drop=True)

court_name_to_id = {}  # court_name -> court_id
court_rows = []
for i, row in unique_courts.iterrows():
    cname   = row['court_name']
    rname   = row['region']
    loc_id  = court_city_map[cname]   # city-level spatial entry for this court
    cid     = i + 1
    court_name_to_id[cname] = cid
    court_rows.append({
        'court_id':         cid,
        'name':             cname,
        'location_id':      loc_id,
        'court_type':       random.choice(court_types),
        'established_year': random.randint(1900, 2005),
    })

df_court = pd.DataFrame(court_rows)
print(f'court: {len(df_court):,} rows')
df_court.head(3)

court: 1,924 rows


,court_id,name,location_id,court_type,established_year
0,1,Инжавинский районный суд,79,Civil,1930
1,2,Мичуринский районный суд,80,Criminal,1918
2,3,Гагаринский районный суд,2067,Administrative,1925


## person + person_status_history (from CSV)

Each CSV row has **3 persons**: offender, victim, judge 
Total persons = 3 x 20531 = 61593

Person IDs are assigned as:
- `offender_id = row_index * 3 + 1`
- `victim_id   = row_index * 3 + 2`
- `judge_pid   = row_index * 3 + 3`

Fields directly from CSV: `gender`, `has_children` (offender only)

In [35]:
# parse year from 'Year 20XX' string
def parse_year(y_str):
    return int(y_str.replace('Year ', '').strip())

def parse_gender(g_str):
    return 'Male' if g_str.startswith('Male') else 'Female'

def parse_has_children(c_str):
    return True if c_str.startswith('1') else False

# build person rows
person_rows = []
psh_rows    = []   # person_status_history
psh_id      = 1

edu_levels   = ['Primary', 'Secondary', 'Bachelor', 'Master', 'PhD']
emp_statuses = ['Employed', 'Unemployed', 'Self-employed', 'Retired', 'Student']

# we will collect per-row person IDs for later use in case tables
offender_ids = []
victim_ids   = []
judge_pids   = []  # person_id of the judge person (not judge_id)

print('Building persons (>>>20k rows)')

for idx, row in df_csv.iterrows():
    year     = parse_year(row['year'])
    base_pid = idx * 3  # 0-indexed row, so person_ids = base_pid+1, +2, +3

    off_pid  = base_pid + 1
    vic_pid  = base_pid + 2
    jud_pid  = base_pid + 3

    offender_ids.append(off_pid)
    victim_ids.append(vic_pid)
    judge_pids.append(jud_pid)

    g_off = parse_gender(row['gender_offender'])
    g_vic = parse_gender(row['gender_victim'])
    g_jud = parse_gender(row['gender_judge'])

    # offender
    person_rows.append({
        'person_id':    off_pid,
        'first_name':   fake_ru.first_name_male() if g_off=='Male' else fake_ru.first_name_female(),
        'last_name':    fake_ru.last_name(),
        'gender':       g_off,
        'birth_date':   fake.date_of_birth(minimum_age=18, maximum_age=60),
        'has_children': parse_has_children(row['children']),   # from CSV
        'nationality':  'Russian',
    })
    # offender status history: employment/income reflects criminal profile
    psh_rows.append({
        'status_id':         psh_id,
        'person_id':         off_pid,
        'education_level':   random.choice(edu_levels),
        'employment_status': random.choice(emp_statuses),
        'income_level':      round(random.uniform(5000, 80000), 2),
        'valid_from':        date(year - random.randint(1, 5), 1, 1),
        'valid_to':          None,
    })
    psh_id += 1

    # victim
    person_rows.append({
        'person_id':    vic_pid,
        'first_name':   fake_ru.first_name_male() if g_vic=='Male' else fake_ru.first_name_female(),
        'last_name':    fake_ru.last_name(),
        'gender':       g_vic,
        'birth_date':   fake.date_of_birth(minimum_age=18, maximum_age=75),
        'has_children': random.choice([True, False]),
        'nationality':  'Russian',
    })
    psh_rows.append({
        'status_id':         psh_id,
        'person_id':         vic_pid,
        'education_level':   random.choice(edu_levels),
        'employment_status': random.choice(emp_statuses),
        'income_level':      round(random.uniform(10000, 120000), 2),
        'valid_from':        date(year - random.randint(1, 5), 1, 1),
        'valid_to':          None,
    })
    psh_id += 1

    # judge
    person_rows.append({
        'person_id':    jud_pid,
        'first_name':   fake_ru.first_name_male() if g_jud=='Male' else fake_ru.first_name_female(),
        'last_name':    fake_ru.last_name(),
        'gender':       g_jud,
        'birth_date':   fake.date_of_birth(minimum_age=35, maximum_age=65),
        'has_children': random.choice([True, False]),
        'nationality':  'Russian',
    })
    psh_rows.append({
        'status_id':         psh_id,
        'person_id':         jud_pid,
        'education_level':   'Master',
        'employment_status': 'Employed',
        'income_level':      round(random.uniform(80000, 300000), 2),
        'valid_from':        date(year - random.randint(5, 15), 1, 1),
        'valid_to':          None,
    })
    psh_id += 1

df_person = pd.DataFrame(person_rows)
df_psh    = pd.DataFrame(psh_rows)
print(f'person:                {len(df_person):,} rows')
print(f'person_status_history: {len(df_psh):,} rows')

Building persons (>>>20k rows)
person:                61,593 rows
person_status_history: 61,593 rows


## judge (from CSV)

Each unique `(court_name, gender_judge)` combination produces one judge record.  
Judge person IDs come from the persons built in Step 6.

In [36]:
specializations = ['Criminal Law', 'Civil Law', 'Corporate', 'Family Law', 'Constitutional']

# one judge per court (using first appearance in CSV for each court)
first_per_court = df_csv.drop_duplicates('court_name').reset_index()

judge_rows = []
judge_person_to_judge_id = {}  # judge person_id → judge_id

for j_idx, row in first_per_court.iterrows():
    orig_idx  = row['index']          # original CSV row index
    jud_pid   = orig_idx * 3 + 3      # judge person_id from Step 6
    year      = parse_year(row['year'])
    court_id  = court_name_to_id[row['court_name']]
    judge_id  = j_idx + 1
    judge_person_to_judge_id[jud_pid] = judge_id

    appt_date = date(max(1990, year - random.randint(2, 20)), random.randint(1, 12), 1)
    term_end  = None  # most judges still active
    if random.random() < 0.15:
        term_end = date(min(2024, year + random.randint(3, 8)), random.randint(1, 12), 1)

    judge_rows.append({
        'judge_id':         judge_id,
        'person_id':        jud_pid,
        'court_id':         court_id,
        'appointment_date': appt_date,
        'term_end':         term_end,
        'specialization':   random.choice(specializations),
    })

df_judge = pd.DataFrame(judge_rows)
print(f'judge: {len(df_judge):,} rows  (one per unique court)')
df_judge.head(3)

judge: 1,924 rows  (one per unique court)


,judge_id,person_id,court_id,appointment_date,term_end,specialization
0,1,3,1,1995-04-01,2020-05-01,Corporate
1,2,6,2,2010-01-01,None,Constitutional
2,3,9,3,2004-06-01,2017-12-01,Corporate


## case_file, crime_act, crime_motive (from CSV)

Each CSV row = one case. All 20,531 rows become real case records.

In [37]:
# guilty plea mapping
def parse_guilty_plea(gp_str):
    """Map CSV string to boolean. Partial = True (some admission), None = no plea."""
    if gp_str.startswith('2'):   return True   # full guilty plea
    if gp_str.startswith('1'):   return True   # partial guilty plea
    return False                               # no guilty plea

# case_status: 'Closed' if trial_end_date set, else 'Open'
# we'll use a simple heuristic: cases from 2013-2018 are Closed, 2019 mix

case_rows      = []
crime_act_rows = []
motive_rows    = []

for idx, row in df_csv.iterrows():
    year       = parse_year(row['year'])
    case_id    = idx + 1
    court_id   = court_name_to_id[row['court_name']]

    # report_date: random date within the case year
    report_date = date(year, random.randint(1, 12), random.randint(1, 28))

    # trial_end: most older cases are closed; 2019 cases 50% still open
    if year < 2019 or random.random() > 0.5:
        trial_end = report_date + timedelta(days=random.randint(60, 540))
        cs_name   = random.choice(['Closed', 'Appealed', 'Dismissed'])
    else:
        trial_end = None
        cs_name   = random.choice(['Open', 'Pending'])

    case_rows.append({
        'case_id':        case_id,
        'court_id':       court_id,
        'year':           year,                            # kept per current DDL
        'case_status_id': status_to_id[cs_name],
        'report_date':    report_date,
        'trial_end_date': trial_end,
        'guilty_plea':    parse_guilty_plea(row['guilty_plea']),
    })

    # crime_act: rel_type from CSV
    rel = row['rel_type']
    # Parse the act type label
    act_type = rel.split(' ', 1)[1] if ' ' in rel else rel
    # hard_binary -> aggravating presence note
    has_aggravating = row['hard_binary'].startswith('1')
    crime_act_rows.append({
        'act_id':      case_id,
        'case_id':     case_id,
        'act_type':    'Homicide',
        'description': act_type + (' — aggravating circumstances present' if has_aggravating else ''),
        'date_of_act': report_date - timedelta(days=random.randint(1, 90)),
    })

    # crime_motive: soft_binary (mitigating)
    has_mitigating = row['soft_binary'].startswith('1')
    motive_rows.append({
        'motive_id':   case_id,
        'act_id':      case_id,
        'description': 'Mitigating circumstances present' if has_mitigating else 'No mitigating circumstances',
    })

df_case      = pd.DataFrame(case_rows)
df_crime_act = pd.DataFrame(crime_act_rows)
df_motive    = pd.DataFrame(motive_rows)
print(f'case_file: {len(df_case):,} rows')
print(f'crime_act: {len(df_crime_act):,} rows')
print(f'crime_motive: {len(df_motive):,} rows')

case_file: 20,531 rows
crime_act: 20,531 rows
crime_motive: 20,531 rows


## sentence (from CSV)

`sentence_period_whole` maps directly to `total_duration_months`.

In [38]:
sent_types = ['Imprisonment', 'Life Sentence', 'Suspended', 'Probation']

sentence_rows = []
for idx, row in df_csv.iterrows():
    case_id  = idx + 1
    duration = int(row['sentence_period_whole']) if not pd.isna(row['sentence_period_whole']) else None

    # infer sentence type from duration
    if duration is None:
        s_type = 'Probation'
    elif duration >= 240:     # 20+ years
        s_type = 'Life Sentence'
    elif duration == 0:
        s_type = 'Suspended'
    else:
        s_type = 'Imprisonment'

    sentence_rows.append({
        'sentence_id':           case_id,
        'case_id':               case_id,
        'total_duration_months': duration,
        'sentence_type':         s_type,
        'parole_eligibility':    duration is not None and duration < 240,
        'fine_amount':           None,   # homicide cases don't have fines
    })

df_sentence = pd.DataFrame(sentence_rows)
print(f'sentence: {len(df_sentence):,} rows')
print(f'Duration range: {df_sentence.total_duration_months.min()}–{df_sentence.total_duration_months.max()} months')

sentence: 20,531 rows
Duration range: 24–372 months


## case_participant + case_judge (from CSV)

Each case gets:
- 1 **Defendant** participant (offender person)
- 1 **Victim** participant
- 1 **case_judge** entry linking the case to the court's judge

In [39]:
cp_rows  = []   # case_participant
cj_rows  = []   # case_judge

# build court -> judge_id lookup
court_to_judge_id = {}
for _, jr in df_judge.iterrows():
    court_to_judge_id[jr['court_id']] = jr['judge_id']

panel_roles = ['Presiding', 'Associate', 'Panel Member']

for idx, row in df_csv.iterrows():
    case_id   = idx + 1
    off_pid   = offender_ids[idx]
    vic_pid   = victim_ids[idx]
    court_id  = court_name_to_id[row['court_name']]
    judge_id  = court_to_judge_id.get(court_id)

    rec_type  = row['recidivism_type']

    # offender as Defendant
    cp_rows.append({
        'case_id':            case_id,
        'person_id':          off_pid,
        'role_id':            ROLE_DEFENDANT,
        'involvement_notes':  rec_type,    # recidivism info stored here
        'participation_level': 'High',
        'remarks':            None,
    })

    # victim
    cp_rows.append({
        'case_id':            case_id,
        'person_id':          vic_pid,
        'role_id':            ROLE_VICTIM,
        'involvement_notes':  None,
        'participation_level': 'High',
        'remarks':            None,
    })

    # case_judge
    if judge_id is not None:
        cj_rows.append({
            'case_id':       case_id,
            'judge_id':      judge_id,
            'role_in_panel': 'Presiding',
        })

df_case_participant = pd.DataFrame(cp_rows)
df_case_judge       = pd.DataFrame(cj_rows)
print(f'case_participant: {len(df_case_participant):,} rows  ({len(df_case_participant)//2:,} offenders + {len(df_case_participant)//2:,} victims)')
print(f'case_judge:       {len(df_case_judge):,} rows')

case_participant: 41,062 rows  (20,531 offenders + 20,531 victims)
case_judge:       20,531 rows


## offender_risk_assessment (from CSV)

`recidivism_type` from CSV drives the risk score range:
- No prior crime $\rightarrow$ low score (0–25)
- Other prior $\rightarrow$ medium (20–55)
- Recidivism degree 1/2/3 $\rightarrow$ high (50–100)

In [40]:
recidivism_score_range = {
    '0 No prior crime records':  (0,  25),
    '1 Other prior crime records': (20, 55),
    '2 Recidivism degree 1':     (45, 70),
    '3 Recidivism degree 2':     (60, 85),
    '4 Recidivism degree 3':     (75, 100),
}

ora_rows = []
for idx, row in df_csv.iterrows():
    off_pid  = offender_ids[idx]
    year     = parse_year(row['year'])
    rec_type = row['recidivism_type']
    lo, hi   = recidivism_score_range.get(rec_type, (0, 100))

    ora_rows.append({
        'assessment_id':   idx + 1,
        'person_id':       off_pid,
        'risk_score':      round(random.uniform(lo, hi), 2),
        'assessment_date': date(year, random.randint(1, 12), random.randint(1, 28)),
    })

df_offender_risk = pd.DataFrame(ora_rows)
print(f'offender_risk_assessment: {len(df_offender_risk):,} rows')
df_offender_risk.groupby(
    df_csv['recidivism_type'].values
)['risk_score'].mean().round(1)

offender_risk_assessment: 20,531 rows


0 No prior crime records       12.7
1 Other prior crime records    37.4
2 Recidivism degree 1          57.7
3 Recidivism degree 2          72.6
4 Recidivism degree 3          87.2
Name: risk_score, dtype: float64

## Synthetic Tables

These tables have no direct CSV/Excel source but are anchored to real regions and courts:
`police_department`, `police_officer`, `prison`, `investigation`, `investigation_officer`,
`evidence`, `appeal`, `appeal_judge`, `case_location`, `sentence_prison`, `police_deployment`

In [41]:
# shared reference: real region IDs from spatial_hierarchy
real_region_ids = list(region_name_to_id.values())   # 77 region location_ids

N = 100   # synthetic rows per table (up from 20 — more realistic with 20k cases)

# police_department
station_types = ['District', 'Regional HQ', 'Precinct', 'Specialized', 'Border']
dept_rows = []
for i in range(1, N+1):
    dept_rows.append({
        'police_id':    i,
        'name':         f'Police Dept {i} — {fake_ru.city()}',
        'location_id':  random.choice(real_region_ids),
        'station_type': random.choice(station_types),
        'staff_count':  random.randint(20, 500),
        'budget':       round(random.uniform(500_000, 10_000_000), 2),
    })
df_police_dept = pd.DataFrame(dept_rows)
print(f'police_department: {len(df_police_dept)} rows')

# police_officer
# need persons for officers - create a fresh batch (IDs after all case persons)
max_case_person_id = len(df_csv) * 3   # 61,593
ranks          = ['Constable', 'Sergeant', 'Lieutenant', 'Captain', 'Major', 'Colonel']
training_levels= ['Basic', 'Intermediate', 'Advanced', 'Specialist']

officer_person_rows = []
officer_rows        = []
for i in range(1, N+1):
    pid    = max_case_person_id + i
    gender = random.choice(['Male', 'Female'])
    officer_person_rows.append({
        'person_id':   pid,
        'first_name':  fake_ru.first_name_male() if gender=='Male' else fake_ru.first_name_female(),
        'last_name':   fake_ru.last_name(),
        'gender':      gender,
        'birth_date':  fake.date_of_birth(minimum_age=22, maximum_age=55),
        'has_children':random.choice([True, False]),
        'nationality': 'Russian',
    })
    joined = fake.date_between(start_date=date(2000,1,1), end_date=date(2020,1,1))
    end    = fake.date_between(start_date=joined+timedelta(days=365), end_date=date(2024,1,1)) \
             if random.random() < 0.2 else None
    officer_rows.append({
        'officer_id':     i,
        'police_id':      random.randint(1, N),
        'person_id':      pid,
        'rank':           random.choice(ranks),
        'date_joined':    joined,
        'end_date':       end,
        'training_level': random.choice(training_levels),
    })
df_officer_persons = pd.DataFrame(officer_person_rows)
df_officer         = pd.DataFrame(officer_rows)
print(f'police_officer persons: {len(df_officer_persons)} rows (person_ids {max_case_person_id+1}–{max_case_person_id+N})')
print(f'police_officer: {len(df_officer)} rows')

# prison
security_levels = ['Minimum', 'Medium', 'Maximum', 'Supermax']
prison_rows = []
for i in range(1, N+1):
    cap = random.randint(200, 2000)
    occ = random.randint(50, cap)
    prison_rows.append({
        'prison_id':         i,
        'name':              f'Correctional Facility {i} — {fake_ru.city()}',
        'location_id':       random.choice(real_region_ids),
        'capacity':          cap,
        'current_occupancy': occ,
        'security_level':    random.choice(security_levels),
        'health_facilities': random.choice(['Basic clinic','Full medical unit','None','Psychiatric ward']),
    })
df_prison = pd.DataFrame(prison_rows)
print(f'prison: {len(df_prison)} rows')

police_department: 100 rows
police_officer persons: 100 rows (person_ids 61594–61693)
police_officer: 100 rows
prison: 100 rows


In [ ]:
# investigation
# sampled from real case_ids
inv_statuses  = ['Ongoing', 'Closed', 'Suspended', 'Cold Case']
sampled_cases = random.sample(range(1, len(df_csv)+1), N)
inv_rows = []

for i, cid in enumerate(sampled_cases, 1):
    start = fake.date_between(start_date=date(2012,1,1), end_date=date(2022,1,1))
    end   = start + timedelta(days=random.randint(30, 730)) if random.random() > 0.4 else None
    inv_rows.append({
        'investigation_id':     i,
        'case_id':              cid,
        'lead_investigator_id': random.randint(1, N),
        'start_date':           start,
        'end_date':             end,
        'investigation_status': random.choice(inv_statuses),
    })
    
df_investigation = pd.DataFrame(inv_rows)
print(f'investigation: {len(df_investigation)} rows')

# evidence
ev_types      = ['Physical','Digital','Documentary','Testimonial','Forensic','CCTV','DNA']
stor_statuses = ['Secured','Destroyed','Missing','Transferred']
ev_rows = []
sampled_cases_ev = random.sample(range(1, len(df_csv)+1), N)
for i, cid in enumerate(sampled_cases_ev, 1):
    ev_rows.append({
        'evidence_id':    i,
        'case_id':        cid,
        'evidence_type':  random.choice(ev_types),
        'description':    fake.sentence(nb_words=8),
        'collected_date': fake.date_between(start_date=date(2012,1,1), end_date=date(2023,1,1)),
        'collected_by':   random.randint(1, N),
        'storage_status': random.choice(stor_statuses),
    })
df_evidence = pd.DataFrame(ev_rows)
print(f'evidence: {len(df_evidence)} rows')

# appeal
appeal_statuses = ['Pending','Granted','Denied','Withdrawn']
# only 'Appealed' cases can have appeals — sample from those
appealed_case_ids = df_case.loc[df_case['case_status_id'] == status_to_id['Appealed'], 'case_id'].tolist()
appeal_sample     = random.sample(appealed_case_ids, min(N, len(appealed_case_ids)))
# need court_ids for appeals — use case's court
case_id_to_court  = dict(zip(df_case['case_id'], df_case['court_id']))
appeal_rows = []
for i, cid in enumerate(appeal_sample, 1):
    filed    = fake.date_between(start_date=date(2013,1,1), end_date=date(2023,6,1))
    decision = filed + timedelta(days=random.randint(30, 365)) if random.random() > 0.3 else None
    appeal_rows.append({
        'appeal_id':     i,
        'case_id':       cid,
        'court_id':      case_id_to_court[cid],
        'date_filed':    filed,
        'result':        random.choice(['Upheld','Overturned','Remanded', None]),
        'appeal_status': random.choice(appeal_statuses),
        'decision_date': decision,
    })
df_appeal = pd.DataFrame(appeal_rows)
print(f'appeal: {len(df_appeal)} rows  (from {len(appealed_case_ids):,} appealed cases)')

# appeal_judge (M:N)
# use real judge_ids from df_judge
real_judge_ids = df_judge['judge_id'].tolist()
aj_used = set()
aj_rows = []
for _, ar in df_appeal.iterrows():
    aid = ar['appeal_id']
    jid = random.choice(real_judge_ids)
    if (aid, jid) not in aj_used:
        aj_used.add((aid, jid))
        aj_rows.append({'appeal_id': aid, 'judge_id': jid})
df_appeal_judge = pd.DataFrame(aj_rows)
print(f'appeal_judge: {len(df_appeal_judge)} rows')

# investigation_officer (M:N)
io_used = set()
io_rows = []
for i in range(N):
    while True:
        inv = random.randint(1, N); o = random.randint(1, N)
        if (inv, o) not in io_used:
            io_used.add((inv, o)); break
    io_rows.append({
        'investigation_id':       inv,
        'officer_id':             o,
        'role_in_investigation':  random.choice(['Lead','Support','Forensics','Surveillance']),
        'hours_assigned':         random.randint(10, 500),
    })
df_inv_officer = pd.DataFrame(io_rows)
print(f'investigation_officer: {len(df_inv_officer)} rows')

# sentence_prison
# link imprisonment sentences to prisons
imprisonment_sids = df_sentence.loc[
    df_sentence['sentence_type'] == 'Imprisonment', 'sentence_id'
].tolist()
sp_sample = random.sample(imprisonment_sids, min(N, len(imprisonment_sids)))
sp_used   = set()
sp_rows   = []
for sid in sp_sample:
    pid = random.randint(1, N)
    entry = fake.date_between(start_date=date(2012,1,1), end_date=date(2022,1,1))
    key   = (sid, pid, entry)
    if key not in sp_used:
        sp_used.add(key)
        release = entry + timedelta(days=random.randint(30, 3650)) \
                  if random.random() > 0.3 else None
        sp_rows.append({
            'sentence_id':  sid,
            'prison_id':    pid,
            'entry_date':   entry,
            'release_date': release,
        })
df_sentence_prison = pd.DataFrame(sp_rows)
print(f'sentence_prison: {len(df_sentence_prison)} rows')

# case_location (M:N)
# link cases to their region via case -> court -> court city -> parent region
cl_rows = []
for idx, row in df_csv.iterrows():
    case_id = idx + 1
    rname   = row['region']
    loc_id  = region_name_to_id.get(rname, region_name_to_id[CATCHALL_REGION])
    cl_rows.append({'case_id': case_id, 'location_id': loc_id})
df_case_location = pd.DataFrame(cl_rows)
print(f'case_location: {len(df_case_location):,} rows')

# police_deployment
pd_used = set()
pd_rows = []
for r_id in real_region_ids:
    for y in [2013, 2014, 2015, 2016, 2017, 2018, 2019]:
        if (r_id, y) not in pd_used:
            pd_used.add((r_id, y))
            pd_rows.append({
                'region_id':         r_id,
                'year':              y,
                'officers_assigned': random.randint(50, 5000),
                'patrol_units':      random.randint(5, 200),
                'budget':            round(random.uniform(1e6, 5e7), 2),
            })
df_deployment = pd.DataFrame(pd_rows)
print(f'police_deployment: {len(df_deployment):,} rows  ({len(real_region_ids)} regions x 7 years)')

investigation: 100 rows
evidence: 100 rows
appeal: 100 rows  (from 6,584 appealed cases)
appeal_judge: 100 rows
investigation_officer: 100 rows
sentence_prison: 100 rows
case_location: 20,531 rows
police_deployment: 539 rows  (77 regions × 7 years)


## Check dataframes

In [43]:
all_tables = {
    'spatial_hierarchy':         df_spatial,
    'case_status':               df_case_status,
    'role':                      df_role,
    'person (cases)':            df_person,
    'person (officers)':         df_officer_persons,
    'person_status_history':     df_psh,
    'police_department':         df_police_dept,
    'police_officer':            df_officer,
    'court':                     df_court,
    'judge':                     df_judge,
    'prison':                    df_prison,
    'case_file':                 df_case,
    'crime_act':                 df_crime_act,
    'crime_motive':              df_motive,
    'investigation':             df_investigation,
    'evidence':                  df_evidence,
    'sentence':                  df_sentence,
    'appeal':                    df_appeal,
    'case_participant':          df_case_participant,
    'case_location':             df_case_location,
    'case_judge':                df_case_judge,
    'appeal_judge':              df_appeal_judge,
    'investigation_officer':     df_inv_officer,
    'sentence_prison':           df_sentence_prison,
    'offender_risk_assessment':  df_offender_risk,
    'police_deployment':         df_deployment,
    'region_statistics':         df_region_stats,
}

for name, df in all_tables.items():
    print(f'\n --- {name} ({len(df):,} rows) ---')
    display(df.head(2))


 --- spatial_hierarchy (2,098 rows) ---


,location_id,name,type,postal_code,parent_location_id
0,1,Russia,country,NaN,NaN
1,2,Алтайский край,region,100000,1.0



 --- case_status (5 rows) ---


,case_status_id,name
0,1,Open
1,2,Closed



 --- role (10 rows) ---


...

,assessment_id,person_id,risk_score,assessment_date
0,1,1,4.64,2013-06-25
1,2,4,91.47,2013-02-28



 --- police_deployment (539 rows) ---


,region_id,year,officers_assigned,patrol_units,budget
0,2,2013,1528,85,29810653.53
1,2,2014,3249,27,39462303.90



 --- region_statistics (608 rows) ---


,region_id,year,mean_population,total_article_105_cases,murders_per_1000,gini_index,unemployment_rate,clearance_rate,mean_january_temperature,mean_june_temperature
0,2,2012,NaN,NaN,0.100,0.377,6.2,NaN,-7.74,12.40
1,2,2013,2394694.0,205.0,0.086,0.379,8.3,7.729,-17.57,15.57


## PostgreSQL

**Large dataset warning:** inserting ~200k+ rows total.  
The `person`, `person_status_history`, `case_file`, `sentence`, `case_participant` tables are large - `chunksize=1000` is used for those.

In [ ]:
def insert_df(conn, table_name, df, chunksize=500):
    df.to_sql(
        name=table_name,
        con=conn,
        schema=SCHEMA,
        if_exists='append',
        index=False,
        method='multi',
        chunksize=chunksize,
    )
    print(f'{len(df):>8,} rows -> {SCHEMA}.{table_name}')

# insert order strictly follows FK dependency chain
try:
    with engine.begin() as conn:
        print('Lookup / reference tables')
        insert_df(conn, 'spatial_hierarchy',       df_spatial)
        insert_df(conn, 'case_status',             df_case_status)
        insert_df(conn, 'role',                    df_role)

        print('Person (all batches before any dependent table)')
        insert_df(conn, 'person',                  df_person,          chunksize=1000)
        insert_df(conn, 'person',                  df_officer_persons, chunksize=500)
        insert_df(conn, 'person_status_history',   df_psh,             chunksize=1000)

        print('Infrastructure')
        insert_df(conn, 'police_department',       df_police_dept)
        insert_df(conn, 'police_officer',          df_officer)
        insert_df(conn, 'court',                   df_court)
        insert_df(conn, 'judge',                   df_judge)
        insert_df(conn, 'prison',                  df_prison)

        print('Case data')
        insert_df(conn, 'case_file',               df_case,            chunksize=1000)
        insert_df(conn, 'crime_act',               df_crime_act,       chunksize=1000)
        insert_df(conn, 'crime_motive',            df_motive,          chunksize=1000)

        print('Investigation & evidence')
        insert_df(conn, 'investigation',           df_investigation)
        insert_df(conn, 'evidence',                df_evidence)

        print('Outcomes')
        insert_df(conn, 'sentence',                df_sentence,        chunksize=1000)
        insert_df(conn, 'appeal',                  df_appeal)

        print('Junction tables')
        insert_df(conn, 'case_participant',        df_case_participant, chunksize=1000)
        insert_df(conn, 'case_location',           df_case_location,   chunksize=1000)
        insert_df(conn, 'case_judge',              df_case_judge,      chunksize=1000)
        insert_df(conn, 'appeal_judge',            df_appeal_judge)
        insert_df(conn, 'investigation_officer',   df_inv_officer)
        insert_df(conn, 'sentence_prison',         df_sentence_prison)

        print('Analytical / supplementary')
        insert_df(conn, 'offender_risk_assessment', df_offender_risk,  chunksize=1000)
        insert_df(conn, 'police_deployment',        df_deployment)
        insert_df(conn, 'region_statistics',        df_region_stats)

except Exception as e:
    print(f'Error: {e}')

Lookup / reference tables


In [ ]:
# engine with connection health checks
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    connect_args={
        'options': f'-csearch_path={SCHEMA}',
        'connect_timeout': 30,
        'keepalives': 1,
        'keepalives_idle': 30,       # send keepalive after 30s idle
        'keepalives_interval': 10,   # retry every 10s
        'keepalives_count': 5,       # drop after 5 failed keepalives
    },
    pool_pre_ping=True,              # test connection before each use
    pool_recycle=300,                # recycle connections every 5 minutes
)

# safer insert with retry and small chunks
def insert_df_safe(table_name, df, chunksize=100, max_retries=3):
    if df.empty:
        print(f'  SKIPPED  {SCHEMA}.{table_name} — DataFrame is empty')
        return 0

    total_rows = len(df)
    inserted   = 0
    n_chunks   = (total_rows + chunksize - 1) // chunksize

    print(f'  → starting {table_name} ({total_rows:,} rows, {n_chunks} chunks)...')

    for chunk_num, start in enumerate(range(0, total_rows, chunksize)):
        chunk   = df.iloc[start : start + chunksize]
        attempt = 0

        while attempt < max_retries:
            try:
                with engine.begin() as conn:
                    chunk.to_sql(
                        name=table_name,
                        con=conn,
                        schema=SCHEMA,
                        if_exists='append',
                        index=False,
                        method='multi',
                        chunksize=len(chunk),
                    )
                inserted += len(chunk)
                break

            except OperationalError as e:
                attempt += 1
                wait = 2 ** attempt
                print(f'Chunk {chunk_num} attempt {attempt} failed: {e}')
                if attempt < max_retries:
                    print(f'     Retrying in {wait}s...')
                    time.sleep(wait)
                    engine.dispose()
                else:
                    print(f'Chunk {chunk_num} failed after {max_retries} attempts')
                    raise

    print(f'{table_name} — {inserted:,} rows inserted')
    return inserted


# updated insert plan using insert_df_safe
# chunksize=100 keeps parameters well under PostgreSQL's 65535 limit
# (100 rows × 7 cols = 700 parameters per statement)

def run_insert_group(label, inserts):
    print(f'\n── {label} {"─" * (60 - len(label))}')
    total = 0
    try:
        for item in inserts:
            table  = item[0]
            df     = item[1]
            # Use explicit chunksize if provided, else default to 100
            chunk  = item[2] if len(item) == 3 else 100
            total += insert_df_safe(table, df, chunksize=chunk)
        print(f'   └─ committed {total:,} rows')
        return total
    except Exception as e:
        print(f'FAILED in group "{label}": {e}')
        raise


INSERT_PLAN = [
    (
        'Lookup / reference tables',
        [
            ('spatial_hierarchy', df_spatial),
            ('case_status',       df_case_status),
            ('role',              df_role),
        ]
    ),
    (
        'Person — all batches',
        [
            ('person',                df_person,          100),
            ('person',                df_officer_persons,  50),
            ('person_status_history', df_psh,             100),
        ]
    ),
    (
        'Infrastructure',
        [
            ('police_department', df_police_dept),
            ('police_officer',    df_officer),
            ('court',             df_court,   50),
            ('judge',             df_judge,   50),
            ('prison',            df_prison),
        ]
    ),
    (
        'Case data',
        [
            ('case_file',    df_case,       100),
            ('crime_act',    df_crime_act,  100),
            ('crime_motive', df_motive,     100),
        ]
    ),
    (
        'Investigation & evidence',
        [
            ('investigation', df_investigation),
            ('evidence',      df_evidence),
        ]
    ),
    (
        'Outcomes',
        [
            ('sentence', df_sentence, 100),
            ('appeal',   df_appeal),
        ]
    ),
    (
        'Junction tables',
        [
            ('case_participant',       df_case_participant, 100),
            ('case_location',          df_case_location,   200),
            ('case_judge',             df_case_judge,      100),
            ('appeal_judge',           df_appeal_judge),
            ('investigation_officer',  df_inv_officer),
            ('sentence_prison',        df_sentence_prison),
        ]
    ),
    (
        'Analytical / supplementary',
        [
            ('offender_risk_assessment', df_offender_risk, 100),
            ('police_deployment',        df_deployment),
            ('region_statistics',        df_region_stats),
        ]
    ),
]

grand_total   = 0
failed_groups = []

for label, inserts in INSERT_PLAN:
    try:
        grand_total += run_insert_group(label, inserts)
    except Exception:
        failed_groups.append(label)

print(f'\n{"═" * 60}')
if failed_groups:
    print(f'Completed with {len(failed_groups)} failed group(s):')
    for g in failed_groups:
        print(f'   • {g}')
else:
    print(f'All groups succeeded — {grand_total:,} total rows inserted')
print(f'{"═" * 60}')


── Lookup / reference tables ───────────────────────────────────
  → starting spatial_hierarchy (2,098 rows, 21 chunks)...


In [ ]:
# fix all float columns that should be integers (FK columns and ID columns)
int_cols = {
    'df_spatial':    ['parent_location_id'],
    'df_case':       ['court_id', 'case_status_id', 'year'],
    'df_crime_act':  ['case_id'],
    'df_motive':     ['act_id'],
    'df_officer':    ['police_id', 'person_id'],
    'df_judge':      ['person_id', 'court_id'],
    'df_psh':        ['person_id'],
    'df_sentence':   ['case_id'],
    'df_investigation': ['case_id', 'lead_investigator_id'],
    'df_evidence':   ['case_id', 'collected_by'],
    'df_appeal':     ['case_id', 'court_id'],
    'df_case_participant': ['case_id', 'person_id', 'role_id'],
    'df_case_location':    ['case_id', 'location_id'],
    'df_case_judge':       ['case_id', 'judge_id'],
    'df_appeal_judge':     ['appeal_id', 'judge_id'],
    'df_inv_officer':      ['investigation_id', 'officer_id'],
    'df_sentence_prison':  ['sentence_id', 'prison_id'],
    'df_offender_risk':    ['person_id'],
    'df_deployment':       ['region_id', 'year'],
    'df_region_stats':     ['region_id', 'year'],
}

for df_name, cols in int_cols.items():
    df = eval(df_name)
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype('Int64')

### **!** error

In [ ]:
# engine
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    connect_args={
        'options': f'-csearch_path={SCHEMA}',
        'connect_timeout': 30,
        'keepalives': 1,
        'keepalives_idle': 30,
        'keepalives_interval': 10,
        'keepalives_count': 5,
    },
    pool_pre_ping=True,
    pool_recycle=300,
)

# COPY-based insert
def insert_df_copy(table_name, df):
    """Uses PostgreSQL COPY protocol — fastest possible insert method."""
    if df.empty:
        print(f'  SKIPPED {table_name}')
        return 0

    print(f' ---> starting {table_name} ({len(df):,} rows)...')

    buffer = io.StringIO()
    df.to_csv(buffer, index=False, header=False)
    buffer.seek(0)

    raw_conn = engine.raw_connection()
    try:
        cursor = raw_conn.cursor()
        cols = ', '.join(df.columns)
        cursor.copy_expert(
            f"COPY {SCHEMA}.{table_name} ({cols}) FROM STDIN WITH CSV",
            buffer
        )
        raw_conn.commit()
        print(f'done  {table_name} — {len(df):,} rows')
        return len(df)
    except Exception as e:
        raw_conn.rollback()
        print(f'FAILED {table_name}: {e}')
        raise
    finally:
        raw_conn.close()


# group runner
def run_insert_group(label, inserts):
    print(f'\n── {label} {"─" * (60 - len(label))}')
    total = 0
    try:
        for item in inserts:
            table = item[0]
            df    = item[1]
            total += insert_df_copy(table, df)
        print(f'   └─ committed {total:,} rows')
        return total
    except Exception as e:
        print(f'FAILED in group "{label}": {e}')
        raise


INSERT_PLAN = [
    (
        'Lookup / reference tables',
        [
            ('spatial_hierarchy', df_spatial),
            ('case_status',       df_case_status),
            ('role',              df_role),
        ]
    ),
    (
        'Person — all batches',
        [
            ('person',                df_person),
            ('person',                df_officer_persons),
            ('person_status_history', df_psh),
        ]
    ),
    (
        'Infrastructure',
        [
            ('police_department', df_police_dept),
            ('police_officer',    df_officer),
            ('court',             df_court),
            ('judge',             df_judge),
            ('prison',            df_prison),
        ]
    ),
    (
        'Case data',
        [
            ('case_file',    df_case),
            ('crime_act',    df_crime_act),
            ('crime_motive', df_motive),
        ]
    ),
    (
        'Investigation & evidence',
        [
            ('investigation', df_investigation),
            ('evidence',      df_evidence),
        ]
    ),
    (
        'Outcomes',
        [
            ('sentence', df_sentence),
            ('appeal',   df_appeal),
        ]
    ),
    (
        'Junction tables',
        [
            ('case_participant',      df_case_participant),
            ('case_location',         df_case_location),
            ('case_judge',            df_case_judge),
            ('appeal_judge',          df_appeal_judge),
            ('investigation_officer', df_inv_officer),
            ('sentence_prison',       df_sentence_prison),
        ]
    ),
    (
        'Analytical / supplementary',
        [
            ('offender_risk_assessment', df_offender_risk),
            ('police_deployment',        df_deployment),
            ('region_statistics',        df_region_stats),
        ]
    ),
]

grand_total   = 0
failed_groups = []

for label, inserts in INSERT_PLAN:
    try:
        grand_total += run_insert_group(label, inserts)
    except Exception:
        failed_groups.append(label)

print(f'\n{"═" * 60}')
if failed_groups:
    print(f'Completed with {len(failed_groups)} failed group(s):')
    for g in failed_groups:
        print(f'   • {g}')
else:
    print(f'All groups succeeded — {grand_total:,} total rows inserted')
print(f'{"═" * 60}')

In [ ]:
with engine.begin() as conn:
    conn.execute(text(f"""
        TRUNCATE TABLE
            {SCHEMA}.offender_risk_assessment,
            {SCHEMA}.person_status_history,
            {SCHEMA}.person
        CASCADE
    """))
    print('Truncated')

fixes = {
    'df_spatial':       ['parent_location_id', 'location_id'],
    'df_case':          ['court_id', 'case_status_id', 'year'],
    'df_crime_act':     ['case_id', 'act_id'],
    'df_motive':        ['act_id', 'motive_id'],
    'df_officer':       ['police_id', 'person_id', 'officer_id'],
    'df_judge':         ['person_id', 'court_id', 'judge_id'],
    'df_psh':           ['person_id', 'status_id'],
    'df_sentence':      ['case_id', 'sentence_id'],
    'df_investigation': ['case_id', 'lead_investigator_id', 'investigation_id'],
    'df_evidence':      ['case_id', 'collected_by', 'evidence_id'],
    'df_appeal':        ['case_id', 'court_id', 'appeal_id'],
    'df_case_participant': ['case_id', 'person_id', 'role_id'],
    'df_case_location':    ['case_id', 'location_id'],
    'df_case_judge':       ['case_id', 'judge_id'],
    'df_appeal_judge':     ['appeal_id', 'judge_id'],
    'df_inv_officer':      ['investigation_id', 'officer_id'],
    'df_sentence_prison':  ['sentence_id', 'prison_id'],
    'df_offender_risk':    ['person_id', 'assessment_id'],
    'df_deployment':       ['region_id', 'year'],
    'df_region_stats':     ['region_id', 'year'],
    'df_police_dept':      ['police_id', 'location_id'],
    'df_prison':           ['prison_id', 'location_id'],
    'df_court':            ['court_id', 'location_id'],
    'df_officer_persons':  ['person_id'],
}

for df_name, cols in fixes.items():
    df = eval(df_name)
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype('Int64')

print('All integer columns fixed')

print('\nparent_location_id sample:')
print(df_spatial['parent_location_id'].head(5).to_string())
print(f'dtype: {df_spatial["parent_location_id"].dtype}')

In [ ]:
grand_total   = 0
failed_groups = []

for label, inserts in INSERT_PLAN:
    try:
        grand_total += run_insert_group(label, inserts)
    except Exception:
        failed_groups.append(label)

print(f'\n{"═" * 60}')
if failed_groups:
    print(f'Completed with {len(failed_groups)} failed group(s):')
    for g in failed_groups:
        print(f'   • {g}')
else:
    print(f'All groups succeeded — {grand_total:,} total rows inserted')
print(f'{"═" * 60}')

In [ ]:
# fix float columns
df_region_stats['mean_population'] = df_region_stats['mean_population'].astype('Int64')
df_region_stats['total_article_105_cases'] = df_region_stats['total_article_105_cases'].astype('Int64')

# insert only the failing table
insert_df_copy('region_statistics', df_region_stats)

In [ ]:
# fix the integer columns in region_statistics
df_region_stats['mean_population'] = df_region_stats['mean_population'].astype('Int64')
df_region_stats['total_article_105_cases'] = df_region_stats['total_article_105_cases'].astype('Int64')

# last failed group
run_insert_group(
    'Analytical / supplementary',
    [
        ('offender_risk_assessment', df_offender_risk),
        ('police_deployment',        df_deployment),
        ('region_statistics',        df_region_stats),
    ]
)

## Check row counts

In [ ]:
table_names = [
    'spatial_hierarchy', 'case_status', 'role',
    'person', 'person_status_history',
    'police_department', 'police_officer',
    'court', 'judge', 'prison',
    'case_file', 'crime_act', 'crime_motive',
    'investigation', 'evidence', 'sentence', 'appeal',
    'case_participant', 'case_location', 'case_judge',
    'appeal_judge', 'investigation_officer',
    'sentence_prison', 'offender_risk_assessment',
    'police_deployment', 'region_statistics',
]

try:
    with engine.connect() as conn:
        total = 0
        for t in table_names:
            count = conn.execute(text(f'SELECT COUNT(*) FROM {SCHEMA}.{t}')).scalar()
            total += count
            print(f'  {t:<35} → {count:>8,} rows')
        print(f'\n  {"TOTAL":<35} → {total:>8,} rows')
except Exception as e:
    print(f'Cannot verify (DB not connected): {e}')
    print('\nLocal DataFrame sizes:')
    total = 0
    for name, df in all_tables.items():
        if '(' not in name:   # skip preview-only labels
            total += len(df)
            print(f'  {name:<35} → {len(df):>8,} rows')
    print(f'  {"TOTAL":<35} -> {total:>8,} rows')

  spatial_hierarchy                   →    2,098 rows
  case_status                         →        5 rows
  role                                →       10 rows
  person                              →   61,693 rows
  person_status_history               →   61,593 rows
  police_department                   →      100 rows
  police_officer                      →      100 rows
  court                               →    1,924 rows
  judge                               →    1,924 rows
  prison                              →      100 rows
  case_file                           →   20,531 rows
  crime_act                           →   20,531 rows
  crime_motive                        →   20,531 rows
  investigation                       →      100 rows
  evidence                            →      100 rows
  sentence                            →   20,531 rows
  appeal                              →      100 rows
  case_participant                    →   41,062 rows
  case_location             

## In case we want to rerun the notebook

In [ ]:
# with engine.begin() as conn:
#     conn.execute(text(f"""
#         TRUNCATE TABLE
#             {SCHEMA}.region_statistics,
#             {SCHEMA}.police_deployment,
#             {SCHEMA}.offender_risk_assessment,
#             {SCHEMA}.sentence_prison,
#             {SCHEMA}.investigation_officer,
#             {SCHEMA}.appeal_judge,
#             {SCHEMA}.case_judge,
#             {SCHEMA}.case_location,
#             {SCHEMA}.case_participant,
#             {SCHEMA}.appeal,
#             {SCHEMA}.sentence,
#             {SCHEMA}.evidence,
#             {SCHEMA}.investigation,
#             {SCHEMA}.crime_motive,
#             {SCHEMA}.crime_act,
#             {SCHEMA}.case_file,
#             {SCHEMA}.prison,
#             {SCHEMA}.judge,
#             {SCHEMA}.court,
#             {SCHEMA}.police_officer,
#             {SCHEMA}.police_department,
#             {SCHEMA}.person_status_history,
#             {SCHEMA}.offender_risk_assessment,
#             {SCHEMA}.person,
#             {SCHEMA}.role,
#             {SCHEMA}.case_status,
#             {SCHEMA}.spatial_hierarchy
#         CASCADE
#     """))
#     print('All tables truncated')